# Tissue Specificity — Step-by-Step Toy Examples

This notebook walks through **three ways to score how tissue-specific a gene is**, using small toy data so you can check every calculation by hand.

We use the same two genes as the slides:

| Gene | Type | What it does |
|---|---|---|
| `MYH7` | **tissue-specific** | a heart muscle protein; mutations cause hypertrophic cardiomyopathy |
| `GAPDH` | **housekeeping** | used by every cell to make energy |

By the end you will have working `fold_change`, `ratio_to_mean`, and `tau` functions that you can apply to any gene.

## 1. Setup: the toy median table

The numbers below are simplified expression values across 8 tissues. They match the slides exactly so you can cross-check.

*(MYH7 is rescaled so its top tissue = 100. GAPDH uses approximate real TPM values.)*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Each column = one gene. Each row = one tissue.
data = pd.DataFrame(
    {
        'MYH7':  [100, 8,   1,   1,   0.5, 0.5, 0.5, 0.5],
        'GAPDH': [480, 520, 610, 540, 490, 460, 470, 500],
    },
    index=['Heart', 'Muscle', 'Brain', 'Liver', 'Lung', 'Kidney', 'Pancreas', 'Skin'],
)

data

### Visualize the two shapes

Before any math, look at the data. The whole story is in these shapes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].bar(data.index, data['MYH7'], color='#d62828')
axes[0].set_title('MYH7 — tissue-specific (one spike)')
axes[0].set_ylabel('Expression (relative)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(data.index, data['GAPDH'], color='#888')
axes[1].set_title('GAPDH — housekeeping (all similar)')
axes[1].set_ylabel('Expression (relative)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

**Observation:** MYH7 has one tall bar (Heart) and seven nearly invisible ones. GAPDH has eight bars all around the same height. A good tissue-specificity score should give MYH7 a *high* number and GAPDH a *low* one.

## 2. Where did those numbers come from?

Real GTEx has **hundreds of donor samples per tissue**, not one number. The values in `data` are *medians* across donors.

Here is a tiny example of going from per-donor measurements to a median table:

In [ ]:
# Per-donor 'raw' data (long format). In real GTEx this would have millions of rows.
raw = pd.DataFrame([
    {'gene': 'MYH7', 'tissue': 'Heart',  'donor': 1, 'expression':  98},
    {'gene': 'MYH7', 'tissue': 'Heart',  'donor': 2, 'expression': 112},
    {'gene': 'MYH7', 'tissue': 'Heart',  'donor': 3, 'expression':  95},
    {'gene': 'MYH7', 'tissue': 'Heart',  'donor': 4, 'expression': 105},
    {'gene': 'MYH7', 'tissue': 'Muscle', 'donor': 1, 'expression':   7},
    {'gene': 'MYH7', 'tissue': 'Muscle', 'donor': 2, 'expression':   9},
    {'gene': 'MYH7', 'tissue': 'Muscle', 'donor': 3, 'expression':   8},
    {'gene': 'MYH7', 'tissue': 'Brain',  'donor': 1, 'expression':   1.2},
    {'gene': 'MYH7', 'tissue': 'Brain',  'donor': 2, 'expression':   0.9},
])
raw

In [ ]:
# Group by (gene, tissue) and take the median across donors.
medians = (
    raw.groupby(['gene', 'tissue'])['expression']
       .median()
       .unstack(level='tissue')   # tissues become columns
)

medians

*That's how `data` above was produced from per-donor measurements.* From here on we work with the median table directly — GTEx publishes it pre-computed.

## 3. Method 1 — Fold change (top ÷ second)

**Question:** how much bigger is the top tissue than the runner-up?

$$\text{fold change} \;=\; \frac{\text{top tissue}}{\text{second-highest tissue}}$$

If one tissue dominates → top ≫ second → large number.
If every tissue is about equal → top ≈ second → number near 1.

In [ ]:
def fold_change(values):
    """Top tissue divided by second-highest tissue."""
    sorted_vals = sorted(values, reverse=True)
    top, second = sorted_vals[0], sorted_vals[1]
    return top / second


# --- Walk through MYH7 ---
vals = sorted(data['MYH7'], reverse=True)
print('MYH7 sorted high → low:', vals)
print(f'  top    = {vals[0]}')
print(f'  second = {vals[1]}')
print(f'  fold change = {vals[0]} / {vals[1]} = {vals[0] / vals[1]:.2f}x')

print()

# --- Walk through GAPDH ---
vals = sorted(data['GAPDH'], reverse=True)
print('GAPDH sorted high → low:', vals)
print(f'  top    = {vals[0]}')
print(f'  second = {vals[1]}')
print(f'  fold change = {vals[0]} / {vals[1]} = {vals[0] / vals[1]:.2f}x')

**Reading the result:**

- **MYH7 = 12.5×** → heart blows out every other tissue. ✓ tissue-specific.
- **GAPDH = 1.13×** → top (Brain 610) barely beats second (Liver 540). The score is essentially saying *'no tissue stands out'* — the housekeeping signature.

## 4. Method 2 — Ratio to mean

**Question:** how much bigger is the top tissue than the *average* tissue?

$$\text{ratio} \;=\; \frac{\text{top tissue}}{\text{mean across all tissues}}$$

Same idea as fold change, but uses **every** tissue (via the mean) instead of just the runner-up.

In [ ]:
def ratio_to_mean(values):
    """Top tissue divided by mean across all tissues."""
    top = max(values)
    mean = sum(values) / len(values)
    return top / mean


# --- MYH7 ---
vals = list(data['MYH7'])
print('MYH7 values:', vals)
print(f'  top  = {max(vals)}')
print(f'  sum  = {sum(vals)}')
print(f'  mean = {sum(vals)} / {len(vals)} = {sum(vals) / len(vals):.4f}')
print(f'  ratio to mean = {max(vals)} / {sum(vals) / len(vals):.4f} = {max(vals) / (sum(vals) / len(vals)):.2f}x')

print()

# --- GAPDH ---
vals = list(data['GAPDH'])
print('GAPDH values:', vals)
print(f'  top  = {max(vals)}')
print(f'  sum  = {sum(vals)}')
print(f'  mean = {sum(vals)} / {len(vals)} = {sum(vals) / len(vals):.4f}')
print(f'  ratio to mean = {max(vals)} / {sum(vals) / len(vals):.4f} = {max(vals) / (sum(vals) / len(vals)):.2f}x')

**Reading the result:**

- **MYH7 = 7.2×** → heart is 7× the typical tissue. The mean (13.9) is small because most tissues are near zero.
- **GAPDH = 1.2×** → top (610) is barely above the mean (509) because every tissue is in the 460–610 band. Housekeeping.

## 5. Method 3 — Tau (τ) score

**Idea:** for each tissue, compute the **gap** to the max:

$$\text{gap}_i \;=\; 1 \;-\; \frac{\text{tissue}_i}{\text{max}}$$

Sum the gaps and divide by *N − 1*:

$$\tau \;=\; \frac{\sum_i \text{gap}_i}{N - 1}$$

Properties:
- If one tissue is the max and the rest are zero → every gap = 1 → τ = 1 (perfectly specific).
- If every tissue equals the max → every gap = 0 → τ = 0 (broadly expressed).
- τ is always between 0 and 1, so it's easy to compare across genes.

In [ ]:
def tau(values):
    """Tau specificity score in [0, 1]. 0 = broadly expressed, 1 = single tissue."""
    values = list(values)
    x_max = max(values)
    if x_max == 0:
        return 0.0  # gene is off everywhere
    gaps = [1 - (v / x_max) for v in values]
    return sum(gaps) / (len(values) - 1)


# --- Walk through MYH7 (max = 100) ---
print('MYH7 — gaps from the max (100):')
x_max = max(data['MYH7'])
gaps = []
for tissue, v in data['MYH7'].items():
    gap = 1 - (v / x_max)
    gaps.append(gap)
    print(f'  {tissue:10s}  1 - {v:>5} / {x_max} = {gap:.4f}')
print(f'  sum of gaps = {sum(gaps):.4f}')
print(f'  tau         = {sum(gaps):.4f} / {len(gaps) - 1} = {sum(gaps) / (len(gaps) - 1):.4f}')

print()

# --- Walk through GAPDH (max = 610) ---
print('GAPDH — gaps from the max (610):')
x_max = max(data['GAPDH'])
gaps = []
for tissue, v in data['GAPDH'].items():
    gap = 1 - (v / x_max)
    gaps.append(gap)
    print(f'  {tissue:10s}  1 - {v:>5} / {x_max} = {gap:.4f}')
print(f'  sum of gaps = {sum(gaps):.4f}')
print(f'  tau         = {sum(gaps):.4f} / {len(gaps) - 1} = {sum(gaps) / (len(gaps) - 1):.4f}')

**Reading the result:**

- **MYH7 τ ≈ 0.98** → almost every tissue has a big gap from heart (because heart is way higher than everything else). Highly tissue-specific.
- **GAPDH τ ≈ 0.19** → every tissue is within ~25% of the max, so every gap is small. The sum stays small and τ stays close to 0. Housekeeping.

## 6. All three scores side by side

Now collect all three methods for both genes:

In [ ]:
scores = pd.DataFrame(
    {
        'fold_change':   [fold_change(data['MYH7']),   fold_change(data['GAPDH'])],
        'ratio_to_mean': [ratio_to_mean(data['MYH7']), ratio_to_mean(data['GAPDH'])],
        'tau':           [tau(data['MYH7']),           tau(data['GAPDH'])],
    },
    index=['MYH7', 'GAPDH'],
)

scores.round(3)

All three methods agree on the conclusion (MYH7 is specific, GAPDH isn't) — they just disagree on the *magnitude*.

Two of them (fold change, ratio to mean) are unbounded ratios. Tau is bounded between 0 and 1, which makes it the easiest to compare across thousands of genes.

## 7. Apply to a small gene 'library'

Add a few more genes and score them all at once — this is exactly what you'd do to GTEx's 20,000-gene matrix.

In [ ]:
library = pd.DataFrame(
    {
        'MYH7':  [100, 8,    1,    1,    0.5, 0.5, 0.5,   0.5],   # heart-specific
        'GAPDH': [480, 520,  610,  540,  490, 460, 470,   500],   # housekeeping
        'NEFM':  [12,  4,    3200, 8,    5,   7,   6,     5],     # brain-specific
        'ALB':   [3,   2,    1,    9100, 4,   5,   3,     5],     # liver-specific
        'DMD':   [140, 2400, 6,    2,    3,   2,   2,     1],     # muscle (some heart too)
        'INS':   [1,   1,    1,    2,    1,   1,   8500,  1],     # pancreas-specific
    },
    index=['Heart', 'Muscle', 'Brain', 'Liver', 'Lung', 'Kidney', 'Pancreas', 'Skin'],
)

# Score every gene with every method, plus record which tissue is the top.
all_scores = pd.DataFrame(
    {
        gene: {
            'fold_change':   fold_change(library[gene]),
            'ratio_to_mean': ratio_to_mean(library[gene]),
            'tau':           tau(library[gene]),
            'top_tissue':    library[gene].idxmax(),
        }
        for gene in library.columns
    }
).T

all_scores

**Reading the table:**

- Genes with **high τ** (close to 1) are tissue-specific. The `top_tissue` column tells you *which* tissue.
- **GAPDH** stands out as the only housekeeping gene — τ ≈ 0.19.
- **DMD** has τ ≈ 0.94 — specific to muscle, but slightly lower than MYH7 (0.98) because DMD is *also* high in heart (140). The score correctly reflects "specific to two tissues, not just one".

### Find tissue-specific genes

Once you have scores, you can **filter** for whatever you care about — e.g., τ > 0.9:

In [ ]:
specific = all_scores[all_scores['tau'] > 0.9]
specific.sort_values('tau', ascending=False)

## 8. Your turn

Predict before you compute:

1. A **perfectly heart-only** gene — Heart = 1000, everything else = 0. What should τ be?
2. A gene with **two equal peaks** — Heart = 100, Brain = 100, everything else = 0. What's τ now? Why?
3. A **noisy, no-pattern** gene — pick eight similar values. Does τ stay near 0?

Edit the cell below, run it, and check if your prediction matched:

In [ ]:
# Your gene — change these eight numbers!
my_gene = [1000, 0, 0, 0, 0, 0, 0, 0]

print(f'fold_change   = {fold_change(my_gene):.4f}')
print(f'ratio_to_mean = {ratio_to_mean(my_gene):.4f}')
print(f'tau           = {tau(my_gene):.4f}')

### Edge case to think about

What happens to `fold_change` if the second-highest value is **zero**? Try it:

```python
fold_change([1000, 0, 0, 0, 0, 0, 0, 0])
```

You'll get a `ZeroDivisionError`. In real data, expression is never exactly zero (there's always a little noise), but a robust pipeline would add a small constant or fall back to tau. This is one reason tau is the workhorse in genomics — it never crashes.

## Recap

| Method | Range | Strength | Weakness |
|---|---|---|---|
| **fold change** | 1 → ∞ | simple, intuitive | only looks at top 2 |
| **ratio to mean** | 1 → ∞ | uses every tissue | mean pulled by outliers |
| **tau** | 0 → 1 | bounded, standard | a bit less intuitive |

All three answer the same question — *'how concentrated is this gene's expression?'* — they just use different math.

**Next step:** load the real GTEx median TPM matrix and apply `tau` to all ~20,000 genes. The notebook code above will work unchanged — just swap `library` for the real matrix.